In [1]:
import pandas as pd
import sys
import os
sys.path.insert(0, '/home/hugo/gitlab.pavlovia/online_test/code')
from my_analysis.data_loader import load_and_aggregate
from my_analysis.utils import calculate_dice
from sklearn.metrics import cohen_kappa_score
import numpy as np
import datetime
import plotly.express as px


In [2]:
data_path = os.path.join('/home/hugo/gitlab.pavlovia/online_test', 'data')
output_dir = os.path.join('/home/hugo/gitlab.pavlovia/online_test', 'outputdata')
print(f"Loading files ...")
df = load_and_aggregate(data_path, cutoff=datetime.datetime(2026, 3, 25), save_data=True)

Loading files ...
Processing file /home/hugo/gitlab.pavlovia/online_test/data/879453_online_test_2026-04-16_15h28.06.260.csv
Processing file /home/hugo/gitlab.pavlovia/online_test/data/864767_online_test_2026-04-16_15h15.08.024.csv
File /home/hugo/gitlab.pavlovia/online_test/data/544729_online_test_2026-03-19_11h46.10.203.csv is older than cutoff date: 2026-03-25 00:00:00
Processing file /home/hugo/gitlab.pavlovia/online_test/data/951008_online_test_2026-04-16_15h39.23.879.csv
Processing file /home/hugo/gitlab.pavlovia/online_test/data/834677_online_test_2026-04-16_15h15.08.331.csv
Processing file /home/hugo/gitlab.pavlovia/online_test/data/273984_online_test_2026-04-09_20h47.09.488.csv
File /home/hugo/gitlab.pavlovia/online_test/data/252796_online_test_2026-02-08_16h57.24.888.csv is older than cutoff date: 2026-03-25 00:00:00
Processing file /home/hugo/gitlab.pavlovia/online_test/data/692878_online_test_2026-03-26_21h22.57.690.csv
Processing file /home/hugo/gitlab.pavlovia/online_test

In [3]:
df = pd.read_csv('data/preload_data.csv')

In [4]:
df['judge_ID'].nunique()

135

In [5]:
df_analysis = df[df['done']].copy()
df_analysis = df_analysis.drop('Unnamed: 0', axis=1 )
df_analysis['clip_uid'] = df_analysis['scene_id'].astype(str) + "_" + df_analysis['player'].astype(str) + "_" + df_analysis['clip'].astype(str)


In [6]:
for scene in df_analysis['scene_id'].unique():
    df_f = df_analysis[df_analysis['scene_id']==scene]
    print('For scene ',scene,' they are', df_f['participant'].nunique(),'judge')
df_analysis['scene_id'].value_counts()

For scene  w3l1s1  they are 21 judge
For scene  w2l3s8  they are 26 judge
For scene  w4l1s7  they are 37 judge
For scene  w5l2s2  they are 29 judge
For scene  w7l3s4  they are 22 judge


scene_id
w5l2s2    11136
w4l1s7    10952
w7l3s4     8800
w3l1s1     8400
w2l3s8     8112
Name: count, dtype: int64

In [7]:
df_analysis.head()

,participant,judge_ID,scene_id,player,question,clip,answer,visualisation_time,clip_path,video_duration,done,clip_uid
0,879453,666851e8b2c5dd35c6aaf99d,w3l1s1,ppo,Q_exH,0,True,29.2107,sourcedata/ppo_mario_ep-20/sub-01/ses-002/beh/...,3.166667,True,w3l1s1_ppo_0
1,879453,666851e8b2c5dd35c6aaf99d,w3l1s1,ppo,Q_exH,1,False,25.1595,sourcedata/ppo_mario_ep-20/sub-01/ses-002/beh/...,2.416667,True,w3l1s1_ppo_1
2,879453,666851e8b2c5dd35c6aaf99d,w3l1s1,ppo,Q_exH,2,False,10.2153,sourcedata/ppo_mario_ep-20/sub-01/ses-002/beh/...,2.416667,True,w3l1s1_ppo_2
3,879453,666851e8b2c5dd35c6aaf99d,w3l1s1,ppo,Q_exH,3,True,13.9311,sourcedata/ppo_mario_ep-20/sub-01/ses-002/beh/...,2.400000,True,w3l1s1_ppo_3
4,879453,666851e8b2c5dd35c6aaf99d,w3l1s1,ppo,Q_exH,4,False,12.5782,sourcedata/ppo_mario_ep-20/sub-01/ses-002/beh/...,2.416667,True,w3l1s1_ppo_4


In [46]:
# --- 3. Agreement Metrics Outliers (< Mean - 1*SD) ---
metric_outliers = set()
judge_metrics = {j: {'percent': [], 'kappa': [], 'dice': []} for j in df_analysis['judge_ID'].unique()}
ref = pd.read_csv('/home/hugo/gitlab.pavlovia/online_test/sourcedata/consensus_reference.csv')
for (scene, player), group in df_analysis.groupby(['scene_id', 'player']):
        # Pivot answers: Index=[question, clip], Columns=judge_ID

        ref_subset = ref[
        (ref['scene_id'] == scene) &
        (ref['player'] == player)
        ]
        pivoted = group.pivot_table(index=['question', 'clip'], columns='judge_ID', values='answer')
        consensus = ref_subset.pivot_table(index=['question', 'clip'], values='answer').iloc[:, 0]
        #print(pivoted)
        #print(consensus)
        for judge in pivoted.columns:
            judge_ans = pivoted[judge]
            aligned = pd.concat([judge_ans, consensus], axis=1, keys=['judge', 'consensus']).dropna()
            v1 = aligned['judge'].astype(int)
            v2 = aligned['consensus'].astype(int)
            judge_metrics[judge]['percent'].append((v1 == v2).mean())
            try:
                k = cohen_kappa_score(v1, v2)
                if np.isnan(k): print('Kappa non dfini pour ',judge)
                judge_metrics[judge]['kappa'].append(k)
            except Exception as e:
                print(f"Kappa error for {judge}, scene {scene}: {e}")
            judge_metrics[judge]['dice'].append(calculate_dice(v1, v2))
# Average metrics per judge
final_judge_metrics = []
for judge, measurements in judge_metrics.items():
        row = {'judge_ID': judge}
        for m in ['percent', 'kappa', 'dice']:
            row[m] = np.mean(measurements[m]) if measurements[m] else np.nan
        final_judge_metrics.append(row)
        
df_metrics = pd.DataFrame(final_judge_metrics)#.set_index('judge_ID')
    
outliers = df_metrics[df_metrics['kappa'] < 0.15]['judge_ID'].tolist()
metric_outliers.update(outliers)
        
#bad_boys = metric_outliers & set(time_outliers.keys())
#outliers_message = outliers_message+[f"{judge} saw less than 80% of video DURATION for {time_outliers[judge]} AND had extrem AGREEMENT measure: {df_metrics.loc[judge, :].to_list()} (percent, kappa, dice)\n" for judge in bad_boys]

print(len(metric_outliers))
metric_outliers

34


{'5e0b3d5dbf3d4e3868abc3ea',
 '5ecc0815f7f5db7c9c661508',
 '5fc6b25adb97a63895d5435f',
 '60551c921afebad605bfd329',
 '60a34e69c627c02da05a4619',
 '611f4665b4d9519095682405',
 '613a6a8a5d67441e6004f50c',
 '6154776b07de67ce15599bd5',
 '615cac6aa7e0f698abb5442c',
 '66156e2820c794a9a34aad79',
 '666b0b42bb6fa2822fa27a32',
 '66ce678bbaaf120fe160a530',
 '673caf90069aa7e3820136cc',
 '67d012f35c929bac3118c6f5',
 '695c0c263b49e67395e42b71',
 '695f9c1b1257661028e09d73',
 '697c89aa6ef612fb57786818',
 '697cae6e93b79bcf6b314046',
 '69807483ab18b3312485a744',
 '6980a4f0faef0e53cdfb97a7',
 '698165f74d737ae8dbb46f7a',
 '698255fa4fcb0cc777631bc2',
 '698345206d65b526d5a59d68',
 '6984783e42cc617bd5d288fd',
 '698b761436a8f2fae4910fb1',
 '698b8d194b513223c9e3d06b',
 '698e02489a2ba0896ae583ca',
 '699a44b0eec33f0efb8996b1',
 '699de824d503ec32677f0e8a',
 '69b2dfa3578726afc01d223c',
 '69bb0909fe9a6471ecdfb3b2',
 '69c62fc9b1c8ff76d5ced051',
 '69d648d33057d63617173b85',
 '69d8a676a584c578712ba8b3'}

In [45]:
px.scatter(df_metrics, x='kappa', y='dice', color='judge_ID')

In [47]:
px.histogram(df_metrics, x='kappa', nbins=40)

In [50]:
time_outliers = set()
print(df_analysis.keys())
df_bad_clips = []
for judge in df_analysis['judge_ID'].unique():

    judge_df = df_analysis[df_analysis['judge_ID'] == judge]
    participant = judge_df['participant'].iloc[0]
        
    # Group by clip to get max vis time for this judge/clip
    judge_clip_stats = judge_df.groupby('clip_uid').agg({
    'visualisation_time': 'max',
    'video_duration': 'first'
    })

    bad_clips = []
    is_time_outlier = False
    for clip_uid, row in judge_clip_stats.iterrows():

        max_vis = row['visualisation_time']
        duration = row['video_duration']
        if pd.notnull(duration) and duration > 0:
            if max_vis < 0.8 * duration:
                    is_time_outlier = True
                    #print('For ', clip_uid, ' visualistion was not sufficient, with: ',duration, 'and max_viz at: ', max_vis)
    
        if is_time_outlier:
            #print(judge, '/',participant ,'is a outilier')
            df_bad_clips.append({
             'judge':judge,
             'participant':participant,
             'bad_clips': clip_uid
            })
        
    if is_time_outlier:
        time_outliers.add(judge)

df_bad_clips = pd.DataFrame(df_bad_clips)
print(df_bad_clips.shape)
#print('Bad boys: ',df_bad_clips['judge'].unique())
print('Nb of Bad boys:',df_bad_clips['judge'].nunique())
print('Nb of bad clips per bad boys: ', df_bad_clips['judge'].value_counts())
#print('Details: ', df_bad_clips['bad_clips'].value_counts())

df_bad_clips.head()

Index(['participant', 'judge_ID', 'scene_id', 'player', 'question', 'clip',
       'answer', 'visualisation_time', 'clip_path', 'video_duration', 'done',
       'clip_uid'],
      dtype='str')
(4034, 3)
Nb of Bad boys: 68
Nb of bad clips per bad boys:  judge
699d5a17143facd69eabf01d    93
695c0c263b49e67395e42b71    88
6984783e42cc617bd5d288fd    87
611f4665b4d9519095682405    87
6977bd8c4a66002ceaa54c1d    86
                            ..
60f1019654f6a6ba79823b19    24
6983546a03db57593e6ddaa9    17
67e1b73ed47c36b4acd92c55    17
69d648d33057d63617173b85     8
66ce678bbaaf120fe160a530     2
Name: count, Length: 68, dtype: int64


,judge,participant,bad_clips
0,68e3cfc22688ddaae48dc6c3,951008,w2l3s8_ppo_27
1,68e3cfc22688ddaae48dc6c3,951008,w2l3s8_ppo_28
2,68e3cfc22688ddaae48dc6c3,951008,w2l3s8_ppo_29
3,68e3cfc22688ddaae48dc6c3,951008,w2l3s8_ppo_3
4,68e3cfc22688ddaae48dc6c3,951008,w2l3s8_ppo_30


In [51]:
import plotly.express as px
px.histogram(df_bad_clips, x='bad_clips')

In [11]:
df_analysis.head()
df_analysis = df_analysis[df_analysis['scene_id']=='w4l1s7']
df_analysis = df_analysis[df_analysis['judge_ID']!='66a9d46ff5d39bbede9f2e92']
print(df_analysis['judge_ID'].nunique())

34


In [14]:
bad_boys = set(df_bad_clips['judge'].unique()) & set(metric_outliers)
bad_boys

{'5e7a0cc0c4ee0904d8d5f154',
 '6151cac776c9ba6b61b3351b',
 '6266aba991affe86ae58ce07',
 '655fe1e57ede6fae3cc26bf0',
 '698074881d252d37dd9bc4d2',
 '69835875bb8198dfcaf4deaa',
 '6997dd1f7023de25b5608079'}

In [15]:
print(df_metrics['judge_ID'].nunique())
df_metrics.head()

34


,judge_ID,percent,kappa,dice
0,6914df8bbecd11a94b6ddb2f,0.689189,0.185918,0.395010
1,6984ac6d0a79fdefd7b87a09,0.807432,0.527710,0.658491
2,5ffcd83b5991d41cc6d5cb40,0.746622,0.474396,0.648003
3,698074881d252d37dd9bc4d2,0.560811,-0.119194,0.180359
4,5dbc09abbc03131b83aeeacd,0.658784,0.111448,0.341088


In [16]:
px.histogram(df_metrics, x='kappa', nbins=30)

In [17]:
px.scatter(df_metrics, x='kappa', y='dice', color='judge_ID')

In [46]:
df_filtered = df_analysis[~df_analysis['judge_ID'].isin(bad_boys)]
df_filtered['judge_ID'].nunique()
df_filtered_hm = df_filtered[df_filtered['player']=='ppo']
df_hm = df_filtered_hm.groupby(['clip', 'question'], as_index=False).agg({
    'answer': 'mean',
    })
df_hm

,clip,question,answer
0,0,Q_eff,0.518519
1,0,Q_exH,0.296296
2,0,Q_exL,0.555556
3,0,Q_ext,0.370370
4,1,Q_eff,0.407407
...,...,...,...
143,35,Q_ext,0.814815
144,36,Q_eff,0.666667
145,36,Q_exH,0.074074
146,36,Q_exL,0.037037


In [50]:
import plotly.express as px
data = df_hm[(df_hm['question']=='Q_exL') | (df_hm['question']=='Q_ext')]
data = data[df_hm['clip']<27]
fig = px.bar(
    data,
    x="clip",
    y="answer",
    color="answer",                 # 👈 continuous color mapping
    color_continuous_scale="Viridis",
    facet_row="question",           # 👈 like your subplots
)

fig.update_layout(
    title="Frequency of responses over each clip",
    height=100 * df['question'].nunique()
)

fig.show()

/tmp/ipykernel_118616/1224353372.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data = data[df_hm['clip']<27]


In [33]:
from my_analysis.metrics import calculate_metrics_panel_sweep
metrics = ['dice', 'percent', 'kappa']
# Panel sizes to test
panel_sizes = [1, 3, 5, 9, 13, 15]
results = calculate_metrics_panel_sweep(df_filtered, metrics, panel_sizes)

Processing Scene: w4l1s7, Player: ppo
  Testing panel size: 1
  Testing panel size: 3


/home/hugo/gitlab.pavlovia/online_test/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/hugo/gitlab.pavlovia/online_test/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:991: RuntimeWarning: invalid value encountered in scalar divide
  k = xp.sum(w_mat * confusion) / xp.sum(w_mat * expected)


  Testing panel size: 5


/home/hugo/gitlab.pavlovia/online_test/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/hugo/gitlab.pavlovia/online_test/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:991: RuntimeWarning: invalid value encountered in scalar divide
  k = xp.sum(w_mat * confusion) / xp.sum(w_mat * expected)
/home/hugo/gitlab.pavlovia/online_test/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/hugo/gitlab.pavlovia/online_test/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:991: RuntimeWarning: invalid value encountered in scalar di

  Testing panel size: 9


/home/hugo/gitlab.pavlovia/online_test/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/hugo/gitlab.pavlovia/online_test/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:991: RuntimeWarning: invalid value encountered in scalar divide
  k = xp.sum(w_mat * confusion) / xp.sum(w_mat * expected)


  Skipping panel size 13: not enough judges (23)
  Skipping panel size 15: not enough judges (23)
Processing Scene: w4l1s7, Player: sub-03
  Testing panel size: 1
  Testing panel size: 3


/home/hugo/gitlab.pavlovia/online_test/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/hugo/gitlab.pavlovia/online_test/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:991: RuntimeWarning: invalid value encountered in scalar divide
  k = xp.sum(w_mat * confusion) / xp.sum(w_mat * expected)


  Testing panel size: 5
  Testing panel size: 9
  Skipping panel size 13: not enough judges (23)
  Skipping panel size 15: not enough judges (23)


In [35]:
from my_analysis.viz import (plot_panel_size_sweep, 
                                          plot_scene_agreement_summary, 
                                          plot_global_agreement_summary, 
                                          plot_correlation_matrix, 
                                          plot_mean_correlation_matrix)
output_dir = os.path.join(os.getcwd(), 'outputdata')
plots_output_path = os.path.join(output_dir, 'results')

plot_scene_agreement_summary(results, output_path=plots_output_path)
plot_correlation_matrix(df_filtered, output_path=plots_output_path)


{'w4l1s7_corr': <Figure size 1000x800 with 2 Axes>}